In [1]:
# %pip install --upgrade bitsandbytes

In [1]:
import os
import gc
import torch
import datasets
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model

2025-09-14 07:22:53.486120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757834573.635031      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757834573.681495      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# 해당 블록만 자기 환경에 맞게 조정하면 그냥 쭉 실행 가능합니당
HF_TOKEN = "REDACTED_HF_TOKEN"

# 사용할 모델 이름
BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
# 데이터 및 모델을 저장할 로컬 경로
PROCESSED_DATA_PATH = "./data"
FINAL_MODEL_PATH = "./finetuned_model"

### 1. data load, preprocess, tokenize

In [3]:
print("Step 1: 데이터셋 로딩 및 분할...")

dataset = load_dataset('zeroshot/twitter-financial-news-sentiment')
dataset['test'] = dataset.pop('validation')

print("\nStep 2: 훈련 데이터 증강...")
doubled_train_dataset = datasets.concatenate_datasets([dataset['train'], dataset['train']])
dataset['train'] = doubled_train_dataset

print("\nStep 3: 데이터 변환 및 토크나이징...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, token=HF_TOKEN)

# Llama-3는 pad 토큰이 없어서 eos 토큰을 pad 토큰으로 사용해야댐
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_seq_length = 512

Step 1: 데이터셋 로딩 및 분할...


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]


Step 2: 훈련 데이터 증강...

Step 3: 데이터 변환 및 토크나이징...


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [5]:
def transform_and_tokenize(example):
    dic = {0:'negative', 1:'positive', 2:'neutral'}
    output = dic[example['label']]
    
    context = (
        f"Instruction: What is the sentiment of this tweet? Please choose an answer from {{negative/neutral/positive}}.\n"
        f"Input: {example['text']}\nAnswer: "
    )

    prompt_ids = tokenizer.encode(context, max_length=max_seq_length, truncation=True)
    target_ids = tokenizer.encode(output, max_length=max_seq_length, truncation=True, add_special_tokens=False)
    
    input_ids = prompt_ids + target_ids + [tokenizer.eos_token_id]
    
    return {
        'input_ids': input_ids,
        'seq_len': len(prompt_ids)
    }

In [6]:
# .map()을 사용하여 전체 데이터셋에 함수 적용
tokenized_dataset = dataset.map(transform_and_tokenize)

# print(f"\nStep 4: 처리된 데이터셋을 로컬에 저장: '{PROCESSED_DATA_PATH}'")
# tokenized_dataset.save_to_disk(PROCESSED_DATA_PATH)

Map:   0%|          | 0/19086 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

### 2. model load, 양자화, LoRA 설정

In [7]:
print("\nStep 5: 모델 로딩 및 양자화 설정...")
# 1. BitsAndBytesConfig 객체 생성
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. from_pretrained 호출 시 quantization_config 인자에 전달
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    token = HF_TOKEN,
    quantization_config=bnb_config, 
)

tokenizer.padding_side = 'left'


Step 5: 모델 로딩 및 양자화 설정...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [8]:
print("\nStep 6: LoRA 설정 적용...")
# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], # Llama-3의 일반적인 타겟 모듈
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)


Step 6: LoRA 설정 적용...


In [9]:
# PEFT 모델로 변환
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


### 4. Train

In [10]:
# print(f"\nStep 7: 저장된 토크나이징 데이터 로드: '{PROCESSED_DATA_PATH}'")
# 저장된 데이터셋 다시 불러오기
# tokenized_dataset = datasets.load_from_disk(PROCESSED_DATA_PATH)

In [11]:
def data_collator(features: list) -> dict:
    len_ids = [len(feature["input_ids"]) for feature in features]
    longest = max(len_ids)
    input_ids = []
    labels_list = []
    
    for ids_l, feature in sorted(zip(len_ids, features), key=lambda x: -x[0]):
        ids = feature["input_ids"]
        seq_len = feature["seq_len"]
        labels = ([-100] * (seq_len - 1) + ids[(seq_len - 1):])
        
        padding_length = longest - ids_l
        ids = ([tokenizer.pad_token_id] * padding_length) + ids
        labels = ([-100] * padding_length) + labels
        input_ids.append(torch.LongTensor(ids))
        labels_list.append(torch.LongTensor(labels))

            
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels_list),
    }

In [12]:
print("\nStep 8: 학습 인자(TrainingArguments) 설정...")
training_args = TrainingArguments(
    output_dir=FINAL_MODEL_PATH,
    label_names=["labels"],
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False
)


Step 8: 학습 인자(TrainingArguments) 설정...


In [13]:
print("\nStep 9: Trainer 초기화 및 학습 시작...")
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

trainer.train()


Step 9: Trainer 초기화 및 학습 시작...


Step,Training Loss,Validation Loss


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/parallel_apply.py", line 96, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/peft/peft_model.py", line 1757, in forward
    return self.base_model(
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^

In [ ]:
print(f"\nStep 10: 최종 모델 저장: '{FINAL_MODEL_PATH}'")
model.save_pretrained(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)